## 01 Data Gathering

#### Purpose
- Gather the data and information in relation to the flood insurance program of US
- Download is done via application program interface (API)

#### Authors
Organized and prepared by Sebastian Carino, Ma. Loisa Celorico, Cyrill Agustin, John Navarro, and Justino Rotairo III. 

- **Date Created:** 2026-11-07
- **Date Modified:** 2026-11-07

### Section 0. Load the libraries and working directory

In [1]:
%load_ext autoreload
%autoreload 2

# Find the repo root from any folder, so `src` can be imported
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import os

import duckdb
import pandas as pd
import pyarrow.parquet as pq
from dotenv import load_dotenv

from src.paths import RAW, RAW_FEMA, RAW_CENSUS, RAW_FRED, ROOT
from src.download import (FEMA_FILES, FRED_SERIES, ACS_YEAR, STATE_FIPS,
                          download_file, download_bulk, fetch_acs)

RAW.mkdir(parents=True, exist_ok=True)

# API key (stored in .env at the repo root; never written in the notebook or committed)
load_dotenv(ROOT / ".env")
KEY = os.environ.get("CENSUS_API_KEY")
assert KEY, "CENSUS_API_KEY not found in .env"

### Section 1. Gather the datasets via API

Our datasets include the following (all downloaded in full, unchanged, into `data/raw/`):
1. NFIP Redacted Policies v3: about 74.7M policy terms (OpenFEMA bulk Parquet)
2. NFIP Redacted Claims v3: about 2.7M claims (OpenFEMA bulk Parquet)
3. Disaster Declarations Summaries (OpenFEMA)
4. NFIP Community Status Book (OpenFEMA)
5. FRED macroeconomic series
6. Census ACS 5-year (2022), tract level

The two NFIP bulk files are large, so they are downloaded with `curl`, which can **resume** an interrupted download. Python's `urllib` is refused by FEMA's server for these files (HTTP 403).

In [2]:
# Download the full NFIP bulk files with curl (resumable; re-run this cell to continue an interrupted download)
# The URLs and file names come from src/download.py, so the notebook and the script never disagree.
for key in ["claims", "policies"]:          # policies is the large one; it may take an hour or more
    url, rel = FEMA_FILES[key]
    download_bulk(url, RAW / rel)

skip (exists): nfip_claims_v3.parquet
skip (exists): nfip_policies_v3.parquet


In [3]:
# Download the small OpenFEMA files
for key in ["disasters", "community"]:
    url, rel = FEMA_FILES[key]
    download_file(url, RAW / rel)

skip (exists): disaster_declarations_summaries_v2.parquet
skip (exists): nfip_community_status_book_v1.parquet


In [4]:
# Download the FRED macroeconomic series (no custom headers: FRED stalls on them)
for sid in FRED_SERIES:
    download_file(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}", RAW_FRED / f"{sid}.csv")

skip (exists): MORTGAGE30US.csv
skip (exists): DGS10.csv
skip (exists): UNRATE.csv
skip (exists): CPIAUCSL.csv
skip (exists): CSUSHPINSA.csv
skip (exists): FEDFUNDS.csv


The American Community Survey (ACS) is a survey the Census Bureau runs every year, sampling about 3.5 million households. It asks about income, housing, age, occupancy and more. The 10-year census only counts people; the ACS is the main source for these details.

**Level:** Census tracts, small neighborhoods of about 4,000 people each, roughly 85,000 across the US. A tract is identified by an 11-digit code (2 state + 3 county + 6 tract), which is the same code as censusGeoid in the NFIP policies.


Code | Name	| Possible use in our models
-----|-----|-----|
B19013_001E|	Median household income	| Ability to keep paying premiums (lapse)
B25077_001E|	Median home value	| Property value and exposure
B01002_001E|	Median age	| Age profile of the neighborhood
B25003_001E|	Occupied housing units	| Size of the housing stock
B25003_002E|	Owner-occupied units	| Owner vs renter share (we derive owner_occupied_share)
B01003_001E|	Total population	| Size of the tract

In [6]:
# Download Census ACS 5-year (2022) tract-level estimates for every state
# (STATE_FIPS = 50 states + DC + Puerto Rico, about 85,000 tracts; a few minutes)
STATES = STATE_FIPS

dest = RAW_CENSUS / f"acs5_{ACS_YEAR}_tract.csv"

# Download again if the saved file is missing any state (e.g. an older Florida-only file)
have = (set(pd.read_csv(dest, usecols=["state"], dtype=str)["state"].str.zfill(2))
        if dest.exists() else set())
missing = sorted(set(STATES) - have)

if not missing:
    print(f"skip (all {len(STATES)} states already in {dest.relative_to(RAW)})")
else:
    print(f"{len(missing)} states missing from the saved file; downloading all {len(STATES)} states ...")
    dest.parent.mkdir(parents=True, exist_ok=True)
    acs = fetch_acs(STATES, KEY)                  # raw values, sentinel codes kept
    acs.to_csv(dest, index=False)                 # replaces the old file only after every state succeeded
    print(f"saved {dest.relative_to(RAW)}: {len(acs):,} tracts, {acs['state'].nunique()} states")

skip (all 52 states already in census/acs5_2022_tract.csv)


In [7]:
# Verify downloads: file sizes and row counts
for f in sorted(RAW.rglob("*")):
    if f.is_file() and not f.name.startswith("."):
        size = f"{f.stat().st_size / 1e6:,.1f} MB"
        rows = f"{pq.ParquetFile(f).metadata.num_rows:,} rows" if f.suffix == ".parquet" else ""
        flag = "  ⚠ incomplete download, delete and re-run" if f.suffix == ".part" else ""
        print(f"{str(f.relative_to(RAW)):55s} {size:>12s}  {rows}{flag}")
# Expected: claims = 2.7M rows, policies = 74.7M rows

census/acs5_2022_tract.csv                                    7.9 MB  
fema/disaster_declarations_summaries_v2.parquet               7.4 MB  70,599 rows
fema/nfip_claims_v3.parquet                                 163.8 MB  2,725,989 rows
fema/nfip_community_status_book_v1.parquet                    1.0 MB  25,133 rows
fema/nfip_policies_v3.parquet                             3,713.0 MB  74,739,461 rows
fred/CPIAUCSL.csv                                             0.0 MB  
fred/CSUSHPINSA.csv                                           0.0 MB  
fred/DGS10.csv                                                0.3 MB  
fred/FEDFUNDS.csv                                             0.0 MB  
fred/MORTGAGE30US.csv                                         0.0 MB  
fred/UNRATE.csv                                               0.0 MB  


#### Loading the full files
The policies file (about 74.7M rows × 87 columns) is too large for pandas in memory. We will query it with DuckDB directly from disk, and use pandas as needed (ideally for small files). 

In [8]:
# Overview of the full NFIP files, computed on disk (nothing large is loaded into memory)
policies_path = RAW_FEMA / "nfip_policies_v3.parquet"
claims_path = RAW_FEMA / "nfip_claims_v3.parquet"

print(duckdb.sql(f"""
    SELECT 'policies' AS dataset, COUNT(*) AS n_rows,
           MIN(policyEffectiveDate) AS first_date, 
           MAX(policyEffectiveDate) AS last_date
    FROM '{policies_path}'
    UNION ALL
    SELECT 'claims', 
            COUNT(*), 
            MIN(dateOfLoss), 
            MAX(dateOfLoss)
    FROM '{claims_path}'
"""))

# Rows per state (top 10)
print(duckdb.sql(f"""
    SELECT propertyState, 
           COUNT(*) AS n_policy_terms
    FROM '{policies_path}'
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
"""))

┌──────────┬──────────┬────────────┬────────────┐
│ dataset  │  n_rows  │ first_date │ last_date  │
│ varchar  │  int64   │    date    │    date    │
├──────────┼──────────┼────────────┼────────────┤
│ policies │ 74739461 │ 2009-01-01 │ 2027-08-05 │
│ claims   │  2725989 │ 1978-01-01 │ 2026-09-07 │
└──────────┴──────────┴────────────┴────────────┘

┌───────────────┬────────────────┐
│ propertyState │ n_policy_terms │
│    varchar    │     int64      │
├───────────────┼────────────────┤
│ FL            │       22377861 │
│ TX            │       11612805 │
│ LA            │        8456525 │
│ CA            │        3953578 │
│ NJ            │        2918984 │
│ SC            │        2715811 │
│ NY            │        2705125 │
│ NC            │        2207699 │
│ VA            │        1668793 │
│ GA            │        1461337 │
└───────────────┴────────────────┘
  10 rows              2 columns



In [9]:
# View one slice into pandas (Florida, 2019-2023)
policies_fl = duckdb.sql(f"""
    SELECT *
    FROM '{policies_path}'
    WHERE propertyState = 'FL'
      AND policyEffectiveDate >= '2019-01-01'
      AND policyEffectiveDate <  '2024-01-01'
""").df()
policies_fl.shape

(5681104, 87)